In [ ]:
!pip install transformers datasets evaluate

# Carga del dataset (mrpc - glue collection)

In [ ]:
from datasets import load_dataset
ds = load_dataset("glue", "mrpc")

In [ ]:
ex = ds["train"][6]
ex

In [ ]:
labels = ds["train"].features["label"]
labels

# Tokenizar usando HF

In [54]:
from transformers import AutoTokenizer

model_id = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

In [27]:
sentence_1 = ds["train"]["sentence1"][0]
sentence_1

'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .'

In [28]:
tokenized_sentence_1 = tokenizer(sentence_1)
tokenized_sentence_1

{'input_ids': [0, 10127, 1001, 6182, 1238, 39, 2138, 2156, 2661, 37, 373, 22, 5, 4562, 22, 2156, 9, 12507, 7018, 23817, 39, 1283, 479, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [29]:
original_sentence = tokenizer.decode(tokenized_sentence_1["input_ids"])
original_sentence

'<s>Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .</s>'

In [36]:
tokenizer("im tired","welcome to the class")

{'input_ids': [0, 757, 7428, 2, 2, 605, 42557, 7, 5, 1380, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [ ]:
tokenizer.convert_ids_to_tokens([0, 757, 7428, 2, 2, 605, 42557, 7, 5, 1380, 2])

In [41]:
def tokenize_fn(texts):
  return tokenizer(texts["sentence1"], texts["sentence2"], truncation=True)

In [ ]:
prepared_ds = ds.map(tokenize_fn, batched=True)
prepared_ds

In [ ]:
prepared_ds["train"]["input_ids"][0]

# Agregar padding

In [50]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Entrenamiento del modelo

In [71]:
import evaluate
import numpy as np

def compute_metrics(pred):
  metric = evaluate.load("glue", "mrpc") # cargamos la métrica especifica de mrpc
  pred_vec, labels = pred # desempaquetamos pred en pred_vec (predicciones del modelo) y labels (etiquetas verdaderas)
  predictions = np.argmax(pred_vec, axis=-1)
  return metric.compute(predictions=predictions, references=labels) #calcula las métricas basándose en los dos valores

# Configuración del modelo

In [66]:
from transformers import AutoModelForSequenceClassification

model_labels = labels.names
model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=len(model_labels),
    id2label={str(i):c for i,c in enumerate(model_labels)},
    label2id={c:str(i) for i,c in enumerate(model_labels)}
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Configuracion del entrenamiento

In [67]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir = "./Rafodev1-distilroberta-base",
    hub_model_id = "Rafodev1/Rafodev1-distilroberta-base",
    evaluation_strategy = "steps",
    num_train_epochs = 3,
    push_to_hub = True,
    load_best_model_at_end = True
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [68]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) Y
Token is valid (permission: fineG

In [72]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=prepared_ds["train"],
    eval_dataset=prepared_ds["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

<ipython-input-72-335343a33de9>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Entrenamiento

In [73]:
train_results = trainer.train()
trainer.save_model()
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)

Step,Training Loss,Validation Loss,Accuracy,F1
500,0.302300,0.923503,0.828431,0.879725
1000,0.257400,0.832193,0.843137,0.890034


events.out.tfevents.1740798004.a136cbd51c50.213.2:   0%|          | 0.00/6.84k [00:00<?, ?B/s]

***** train metrics *****
  epoch                    =        3.0
  total_flos               =   191360GF
  train_loss               =     0.2386
  train_runtime            = 0:02:35.24
  train_samples_per_second =     70.881
  train_steps_per_second   =       8.87


# Evaluación


In [76]:
metrics = trainer.evaluate(prepared_ds["test"])
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)

***** eval metrics *****
  epoch                   =        3.0
  eval_accuracy           =     0.8359
  eval_f1                 =     0.8798
  eval_loss               =     0.8613
  eval_runtime            = 0:00:05.38
  eval_samples_per_second =    320.315
  eval_steps_per_second   =     40.109


# Exportamos el modelo

In [109]:
import os

path = "./test-model"

model.save_pretrained(path)
tokenizer.save_pretrained(path)

('./test-model/tokenizer_config.json',
 './test-model/special_tokens_map.json',
 './test-model/vocab.json',
 './test-model/merges.txt',
 './test-model/added_tokens.json',
 './test-model/tokenizer.json')

# Probando el modelo

In [121]:
import torch

sentence1 = "The cat jumped over the fence."
sentence2 = "He arrived late to the meeting."

inputs = tokenizer(sentence1, sentence2, return_tensors="pt", padding=True, truncation=True)
inputs = {k: v.to("cuda")  for k,v in inputs.items()}

model = model.to("cuda")

In [122]:
with torch.no_grad():
  outputs = model(**inputs)
  predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)

  label_names=["no equivalente", "equivalente"]
  score = predictions[0][1].item()

  print(f"Prediccion: {label_names[ 1 if score > 0.5 else 0]}")

Prediccion: no equivalente
